# Vígil.ia — RT-DETR (detecção) · treino em 3 etapas

Plano:
1. **Base** — RT-DETR nas 12.528 imagens do dataset original.
2. **Fine-tune** — nas suas fotos reais (celular).
3. **Teste em vídeo** — ver onde erra → justifica a etapa de treino em vídeo (anotação real).

### O pulo do gato: pseudo-rótulo
O dataset original é de **classificação** (pastas por classe, 1 grão, **sem caixas**).
RT-DETR é **detector** — exige caixas. Então a gente **gera as caixas automaticamente**
com **Otsu** (mesmo algoritmo do `crop_single_grain`): cada imagem tem 1 grão em fundo
escuro → Otsu acha o grão → vira 1 caixa, e a classe vem da pasta. Zero anotação manual.

**Caveat honesto:** esse base aprende a ver **um grão isolado**, não vários amontoados.
O teste em vídeo (etapa 3) provavelmente ainda vai falhar em cenas densas — e é **isso
que queremos observar**, é o que motiva a anotação de vídeo depois. O `mosaic` (cola 4
imagens numa) ajuda a simular multi-grão de graça durante o treino.

> Rodar no Colab com **GPU A100**. O YOLO11s-cls atual continua intacto pro modo foto.

## 1. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU! Troque o ambiente para A100.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Caminhos dos dados
Ajuste os caminhos do seu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset de CLASSIFICAÇÃO (12.5k): pastas train/valid/test com subpastas por classe
CLS_BASE = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)'

# Suas fotos reais (fine-tune): pastas com nomes PT das classes (busca recursiva)
REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

# Vídeo de teste (etapa 3): grave um vídeo curto com VÁRIOS grãos, fundo escuro
VIDEO_TESTE = '/content/drive/MyDrive/teste_soja.mp4'

## 3. Pseudo-rotulador (Otsu → caixa) + builder do dataset de detecção
Transforma as pastas de classificação num dataset YOLO-detect (imagens + labels + data.yaml).

In [ ]:
import os, glob, hashlib, unicodedata, cv2, yaml

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def build_detection_set(items, out_dir):
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    kept = skipped = 0
    per = {}
    for i, (path, cls, sp) in enumerate(items):
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        box = otsu_box(img)
        if box is None:
            skipped += 1; continue
        stem = f'{sp}_{i:06d}'
        ext = os.path.splitext(path)[1].lower()
        dst = f'{out_dir}/images/{sp}/{stem}{ext}'
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)
        cx, cy, ww, hh = box
        with open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w') as f:
            f.write(f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}')
        kept += 1
        per[(sp, cls)] = per.get((sp, cls), 0) + 1
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    print(f'kept={kept}  skipped={skipped}')
    for sp in ('train', 'val', 'test'):
        row = {NAMES[c]: per.get((sp, c), 0) for c in range(5)}
        if sum(row.values()):
            print(sp, row)
    return f'{out_dir}/data.yaml'

def collect_base(base_dir):
    items = []
    for sp_dir, sp in (('train', 'train'), ('valid', 'val'), ('val', 'val'), ('test', 'test')):
        d = os.path.join(base_dir, sp_dir)
        if not os.path.isdir(d):
            continue
        for folder in sorted(os.listdir(d)):
            cls = class_of(folder)
            if cls is None:
                continue
            for p in glob.glob(os.path.join(d, folder, '*')):
                if p.lower().endswith(IMG_EXT):
                    items.append((p, cls, sp))
    return items

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split('/')):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    sp = 'val' if (h % 100) < val_frac * 100 else 'train'
                    items.append((p, cls, sp))
    return items

In [ ]:
# Constrói o dataset base (12.5k) com caixas pseudo-rotuladas
BASE_YAML = build_detection_set(collect_base(CLS_BASE), '/content/soja_det_base')
print('data.yaml ->', BASE_YAML)

In [ ]:
# Sanidade visual: confere se as caixas do Otsu estão certas
import matplotlib.pyplot as plt
sample = glob.glob('/content/soja_det_base/images/train/*')[:6]
plt.figure(figsize=(12, 7))
for i, p in enumerate(sample):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    parts = open(lbl).read().split()
    cls, cx, cy, ww, hh = int(parts[0]), *[float(x) for x in parts[1:]]
    x1, y1 = int((cx - ww / 2) * w), int((cy - hh / 2) * h)
    x2, y2 = int((cx + ww / 2) * w), int((cy + hh / 2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax = plt.subplot(2, 3, i + 1); ax.imshow(img); ax.set_title(NAMES[cls]); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Etapa 1 — RT-DETR base (12.5k)
`rtdetr-l` pré-treinado COCO → transfer learning. `mosaic` simula multi-grão.

In [ ]:
from ultralytics import RTDETR

BATCH = 16   # A100 40GB. Numa A100 80GB dá pra subir p/ 32. Se der OOM, baixe.

model = RTDETR('rtdetr-l.pt')
model.train(
    data=BASE_YAML, epochs=80, imgsz=640, batch=BATCH, device=0, seed=42,
    optimizer='AdamW', lr0=1e-4, patience=20, close_mosaic=10, amp=True,
    cache=False, workers=8,
    mosaic=1.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    degrees=15, translate=0.1, scale=0.5, fliplr=0.5, flipud=0.5,
    project='runs_rtdetr', name='base_12k', exist_ok=True,
)

In [ ]:
# mAP do base no split de teste
r = RTDETR('runs_rtdetr/base_12k/weights/best.pt').val(
    data=BASE_YAML, split='test', imgsz=640, device=0)
print(f'BASE  mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')

## 5. Etapa 2 — Fine-tune nas suas fotos reais
lr menor (não esquecer o que aprendeu no base). Mesmo pseudo-rótulo Otsu.

In [ ]:
REAL_YAML = build_detection_set(collect_real(REAL_SRCS), '/content/soja_det_real')

ft = RTDETR('runs_rtdetr/base_12k/weights/best.pt')
ft.train(
    data=REAL_YAML, epochs=60, imgsz=640, batch=BATCH, device=0, seed=42,
    optimizer='AdamW', lr0=5e-5, patience=25, close_mosaic=10, amp=True,
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_rtdetr', name='ft_real', exist_ok=True,
)

## 6. Etapa 3 — Teste em vídeo (o momento da verdade)
Roda no vídeo real e salva anotado. É aqui que se vê se precisa (e quanto) do treino em vídeo.

In [ ]:
best = RTDETR('runs_rtdetr/ft_real/weights/best.pt')
best.predict(source=VIDEO_TESTE, conf=0.25, save=True,
             project='runs_rtdetr', name='pred_video', exist_ok=True)
print('Vídeo anotado -> runs_rtdetr/pred_video/')

# QC premium no 1º frame: % intacto >= 90% -> Aprovado
frame = next(iter(best.predict(source=VIDEO_TESTE, conf=0.25, stream=True)))
ids = frame.boxes.cls.cpu().numpy().astype(int)
total = len(ids)
premium = int((ids == NAMES.index('intact')).sum())
pct = premium / total if total else 0
print(f'{total} graos | {premium} intactos ({pct*100:.0f}%) -> ' +
      ('APROVADO' if pct >= 0.9 else 'REPROVADO'))

## 7. Exportar
`.pt` pro servidor (HF Space) e `.onnx` caso a gente teste no browser depois.

In [ ]:
import shutil
shutil.copy('runs_rtdetr/ft_real/weights/best.pt', 'soja_rtdetr_finetuned.pt')
RTDETR('soja_rtdetr_finetuned.pt').export(format='onnx', imgsz=640, opset=16, simplify=True)
print('Gerado: soja_rtdetr_finetuned.pt (+ .onnx)')

from google.colab import files
files.download('soja_rtdetr_finetuned.pt')